In [1]:
import GraphPCA as sg
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import squidpy as sq
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances as pair
from sklearn.metrics import adjusted_rand_score as ari_score

/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score

In [3]:
import glob
data_dir_full = []
for data_dir in glob.glob("/data/hoan/spatial_transcriptomics/data/SDMBench/Data/*.h5ad", recursive=True):
    # print(data_dir)
    data_dir_full.append(data_dir)
for data_dir in glob.glob("/data/hoan/spatial_transcriptomics/data/SDMBench/Data/*/*.h5ad", recursive=True):
    if '378k' not in data_dir:
        # print(data_dir)
        data_dir_full.append(data_dir)

In [4]:
output_folder = '/data/hoan/spatial_transcriptomics/GraphBG-analysis/output/graphpca/'

In [9]:
import time
start_time = time.perf_counter()

data_dir = '/data/hoan/spatial_transcriptomics/data/SDMBench/Data/378k/BrainAgingSpatialAtlas_MERFISH.h5ad'
adata=sc.read_h5ad(data_dir)
adata.obsm['spatial'] = adata.obsm['spatial_coords']
adata = adata[:10000,:].copy()
print(adata.shape)

# # Load and filter out NaN labels
# adata = sc.read_h5ad(data_dir)
# adata = adata[~adata.obs['Region'].isna()].copy()

true_labels = pd.factorize(adata.obs['tissue'])[0]
adata.obs['x'] = adata.obsm['spatial'][:, 0]
adata.obs['y'] = adata.obsm['spatial'][:, 1]
adata.X = adata.raw.X
adata.var_names_make_unique()
# sc.pp.filter_genes(adata, min_cells=0)
sc.experimental.pp.normalize_pearson_residuals(adata)
sc.pp.scale(adata)
x_array=adata.obs["x"].tolist()
y_array=adata.obs["y"].tolist()
location=np.array([x_array, y_array]).T.astype(np.float32)
print('Run GPCA')
Z,W = sg.Run_GPCA(adata, location=location, n_components = 20, method = "knn",  n_neighbors = 7, _lambda =0.5,
                save_reconstruction=False, max_iter = 1000)

print('Run clustering')
adata.obsm["GraphPCA"] = Z

estimator = KMeans(n_clusters=len(np.unique(true_labels)))
res = estimator.fit(Z[:,:])
lable_pred=res.labels_
adata.obs["GPCA_pred"]= lable_pred
adata.obs["GPCA_pred"] = adata.obs["GPCA_pred"].astype('category')
refined_pred=sg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["GPCA_pred"].tolist(), dis= pair(location),
                       shape="generic",neighbor_num=7)
adata.obs["GPCA_pred"]= refined_pred
adata.obs["GPCA_pred"] = adata.obs["GPCA_pred"].astype('category')

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.4f} seconds")

(10000, 374)
Run GPCA


C++ module 'gpca_cpp' not found. Falling back to Python 'iterative' mode.


Run clustering
Execution time: 85.6561 seconds


In [6]:
## Full data
# --------------------------------------------------------------------------
# MemoryError                               Traceback (most recent call last)
# Cell In[7], line 35
#      31 res = estimator.fit(Z[:,:])
#      32 lable_pred=res.labels_
#      33 adata.obs["GPCA_pred"]= lable_pred
#      34 adata.obs["GPCA_pred"] = adata.obs["GPCA_pred"].astype('category')
# ---> 35 refined_pred=sg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["GPCA_pred"].tolist(), dis= pair(location),
#      36                        shape="generic",neighbor_num=7)
#      37 adata.obs["GPCA_pred"]= refined_pred
#      38 adata.obs["GPCA_pred"] = adata.obs["GPCA_pred"].astype('category')

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:218, in validate_params.<locals>.decorator.<locals>.wrapper(*args, **kwargs)
#     212 try:
#     213     with config_context(
#     214         skip_parameter_validation=(
#     215             prefer_skip_nested_validation or global_skip_validation
#     216         )
#     217     ):
# --> 218         return func(*args, **kwargs)
#     219 except InvalidParameterError as e:
#     220     # When the function is just a wrapper around an estimator, we allow
#     221     # the function to delegate validation to the estimator, but we replace
#     222     # the name of the estimator by the name of the function in the error
#     223     # message to avoid confusion.
#     224     msg = re.sub(
#     225         r"parameter of \w+ must be",
#     226         f"parameter of {func.__qualname__} must be",
#     227         str(e),
#     228     )

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2488, in pairwise_distances(X, Y, metric, n_jobs, ensure_all_finite, **kwds)
#    2485         return distance.squareform(distance.pdist(X, metric=metric, **kwds))
#    2486     func = partial(distance.cdist, metric=metric, **kwds)
# -> 2488 return _parallel_pairwise(X, Y, func, n_jobs, **kwds)

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:1975, in _parallel_pairwise(X, Y, func, n_jobs, **kwds)
#    1972     Y = X
#    1974 if effective_n_jobs(n_jobs) == 1:
# -> 1975     return func(X, Y, **kwds)
#    1977 # enforce a threading backend to prevent data communication overhead
#    1978 fd = delayed(_transposed_dist_wrapper)

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:191, in validate_params.<locals>.decorator.<locals>.wrapper(*args, **kwargs)
#     189 global_skip_validation = get_config()["skip_parameter_validation"]
#     190 if global_skip_validation:
# --> 191     return func(*args, **kwargs)
#     193 func_sig = signature(func)
#     195 # Map *args/**kwargs to the function signature

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:373, in euclidean_distances(X, Y, Y_norm_squared, squared, X_norm_squared)
#     367     if Y_norm_squared.shape != (1, Y.shape[0]):
#     368         raise ValueError(
#     369             f"Incompatible dimensions for Y of shape {Y.shape} and "
#     370             f"Y_norm_squared of shape {original_shape}."
#     371         )
# --> 373 return _euclidean_distances(X, Y, X_norm_squared, Y_norm_squared, squared)

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:406, in _euclidean_distances(X, Y, X_norm_squared, Y_norm_squared, squared)
#     401         YY = None
#     403 if X.dtype == xp.float32 or Y.dtype == xp.float32:
#     404     # To minimize precision issues with float32, we compute the distance
#     405     # matrix on chunks of X and Y upcast to float64
# --> 406     distances = _euclidean_distances_upcast(X, XX, Y, YY)
#     407 else:
#     408     # if dtype is already float64, no need to chunk and upcast
#     409     distances = -2 * safe_sparse_dot(X, Y.T, dense_output=True)

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:581, in _euclidean_distances_upcast(X, XX, Y, YY, batch_size)
#     578 n_samples_Y = Y.shape[0]
#     579 n_features = X.shape[1]
# --> 581 distances = xp.empty((n_samples_X, n_samples_Y), dtype=xp.float32, device=device_)
#     583 if batch_size is None:
#     584     x_density = X.nnz / np.prod(X.shape) if issparse(X) else 1

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/externals/array_api_compat/_internal.py:35, in get_xp.<locals>.inner.<locals>.wrapped_f(*args, **kwargs)
#      33 @wraps(f)
#      34 def wrapped_f(*args: object, **kwargs: object) -> object:
# ---> 35     return f(*args, xp=xp, **kwargs)

# File ~/miniconda3/envs/biopy311/lib/python3.11/site-packages/sklearn/externals/array_api_compat/common/_aliases.py:49, in empty(shape, xp, dtype, device, **kwargs)
#      40 def empty(
#      41     shape: int | tuple[int, ...],
#      42     xp: Namespace,
#    (...)     46     **kwargs: object,
#      47 ) -> Array:
#      48     _check_device(xp, device)
# ---> 49     return xp.empty(shape, dtype=dtype, **kwargs)

# MemoryError: Unable to allocate 535. GiB for an array with shape (378918, 378918) and data type float32